# `mdpp` Example: Pairwise Distances

Track specific atom-pair and inter-group distances over time.

- `compute_distances`
- `compute_minimum_distance`

Use this notebook after the IO/alignment workflow.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.utils import auto_ticks

from mdpp.analysis.distance import compute_distances, compute_minimum_distance
from mdpp.core.trajectory import load_trajectory, select_atom_indices
from mdpp.plots.timeseries import plot_distances

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Pairwise Atom Distances


In [ ]:
# Track distances between specific atom pairs (e.g., CA atoms)
ca_indices = select_atom_indices(traj.topology, "name CA")
pairs = np.array([[ca_indices[0], ca_indices[-1]], [ca_indices[10], ca_indices[50]]])

result = compute_distances(traj, atom_pairs=pairs)
print(f"Pairs monitored: {result.atom_pairs.shape[0]}")
print(f"Frames: {result.distances_nm.shape[0]}")

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
plot_distances(result, ax=ax, pair_labels=["CA_first-CA_last", "CA_10-CA_50"])
ax.set_title("Pairwise CA Distances")
auto_ticks(ax)
fig.tight_layout()

## Minimum Distance Between Groups


In [ ]:
# Minimum distance between two residue groups
min_dist = compute_minimum_distance(traj, group1="resid 10", group2="resid 50")

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
plot_distances(min_dist, ax=ax, pair_labels=["min(resid 10 - resid 50)"])
ax.set_title("Minimum Inter-Residue Distance")
auto_ticks(ax)
fig.tight_layout()

## Backend Selection

Five backends are available for pairwise distance computation:

| Backend | Device | PBC | Dependency |
|---------|--------|-----|------------|
| `mdtraj` (default) | CPU (1 thread) | Yes | built-in |
| `numba` | CPU (all cores) | No | built-in (numba) |
| `cupy` | GPU (CUDA) | No | `pip install cupy-cuda12x` |
| `torch` | GPU (CUDA) / CPU | No | `pip install torch` |
| `jax` | GPU / TPU / CPU | No | `pip install jax[cuda12]` |

Only `mdtraj` supports periodic boundary conditions. Use `numba` for
faster non-periodic computation on multi-core machines.


In [ ]:
# Numba backend: faster for non-periodic systems
result_numba = compute_distances(traj, atom_pairs=pairs, backend="numba", periodic=False)
np.testing.assert_allclose(result.distances_nm, result_numba.distances_nm, atol=1e-5)
print("mdtraj and numba backends agree.")